## 2. Mathematical Background

This section covers the three mathematical pillars of this project:

1. **Graph Theory** — how we model the mountain terrain
2. **Naismith's Rule** — how we estimate hiking time from distance and elevation
3. **Dijkstra's Algorithm** — how we find the optimal path in a graph

---

### 2.1 Graph Theory

A **graph** is a mathematical structure used to model pairwise relationships between objects.
Formally, a graph is defined as:

$$G = (V, E)$$

Where:
- $V$ = a set of **vertices** (also called nodes)
- $E$ = a set of **edges** — pairs $(u, v)$ representing connections between vertices

In our project:
- Each **GPS point** (latitude, longitude, elevation) is a **vertex**
- Each **connection between two consecutive GPS points** is an **edge**

#### Weighted Graphs

A **weighted graph** assigns a numerical value (weight) to each edge:

$$G = (V, E, w)$$

Where $w: E \rightarrow \mathbb{R}^+$ is the **weight function**.

We will define **three different weight functions** for the same graph:

| Weight function | Meaning |
|---|---|
| $w_d(u,v)$ — Haversine distance | Minimize total distance (km) |
| $w_e(u,v)$ — Elevation gain | Minimize total ascent (m) |
| $w_t(u,v)$ — Naismith time | Minimize estimated hiking time |

#### Haversine Distance

Since GPS coordinates are on a sphere, we use the **Haversine formula**:

$$d = 2R \cdot \arctan\left(\sqrt{\frac{a}{1-a}}\right)$$

$$a = \sin^2\!\left(\frac{\Delta\phi}{2}\right) + \cos\phi_1 \cdot \cos\phi_2 \cdot \sin^2\!\left(\frac{\Delta\lambda}{2}\right)$$

Where $R = 6{,}371{,}000$ m (Earth radius), $\phi$ = latitude, $\lambda$ = longitude.

---

### 2.2 Naismith's Rule

**Naismith's Rule** estimates hiking time, proposed by William W. Naismith in 1892:

$$T = \frac{d}{5} + \frac{h}{600}$$

Where:
- $T$ = estimated time (hours)
- $d$ = horizontal distance (km)
- $h$ = total elevation **gain** (m)

#### Edge-level Naismith weight

For a single edge between two GPS points $u$ and $v$:

$$w_t(u,v) = \frac{d(u,v)}{5000} + \frac{\max(0,\; e_v - e_u)}{600}$$

#### Applied to our tracks

| Track | Distance | Gain | Naismith Time |
|---|---|---|---|
| Rila Monastery → Seven Lakes | 37.47 km | +2,947 m | **12.4 h** |
| Vihren Peak (Pirin) | 8.87 km | +1,053 m | **3.5 h** |

#### Tobler's Hiking Function (comparison)

A more refined alternative that also penalizes steep descents:

$$v = 6 \cdot e^{-3.5 \cdot |s + 0.05|}$$

Where $v$ is speed (km/h) and $s = \tan(\text{slope angle})$.

---

### 2.3 Dijkstra's Algorithm

Published by Edsger W. Dijkstra in 1959. Finds the **shortest path** from source $s$ to all vertices:

$$\delta(s, v) = \min_{p: s \rightsquigarrow v} \sum_{e \in p} w(e)$$

#### Algorithm Steps

```
1. Initialize: dist[s] = 0,  dist[v] = inf for all v != s
2. Add all vertices to priority queue Q
3. While Q is not empty:
   a. Extract vertex u with minimum dist[u]
   b. For each neighbor v of u:
      candidate = dist[u] + w(u, v)
      if candidate < dist[v]:
         dist[v] = candidate
         prev[v] = u
4. Return dist[] and prev[]
```

#### Time Complexity

Using a **min-heap** priority queue:

$$O\bigl((|V| + |E|) \log |V|\bigr)$$

For Rila: $|V| = 3{,}495$ nodes, $|E| \approx 3{,}494$ edges — very fast.

#### Why Dijkstra works here

All edge weights must be **non-negative** — satisfied by all three weight functions:
- Distance $\geq 0$ ✅
- Elevation gain uses $\max(0, \Delta e) \geq 0$ ✅
- Naismith time $\geq 0$ ✅

---

In [ ]:
# ============================================================
# Section 2 — Mathematical Functions
# ============================================================

import math
import numpy as np
import matplotlib.pyplot as plt


def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance between two GPS points (returns meters)."""
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi   = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (math.sin(dphi/2)**2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def naismith_weight(dist_m, elev1, elev2):
    """Naismith hiking time for one edge (returns hours)."""
    elev_gain = max(0, elev2 - elev1)
    return (dist_m / 1000) / 5 + elev_gain / 600


def tobler_weight(dist_m, elev1, elev2):
    """Tobler hiking time for one edge (returns hours)."""
    if dist_m == 0:
        return 0
    slope = (elev2 - elev1) / dist_m
    speed = 6 * math.exp(-3.5 * abs(slope + 0.05))
    return (dist_m / 1000) / speed


# --- Verify with our real track data ---
print('Naismith Verification')
print('=' * 45)
tracks = [
    ('Rila (Monastery -> Seven Lakes)', 37.47, 2947),
    ('Vihren Peak (Pirin)',              8.87,  1053),
]
for name, dist_km, gain in tracks:
    T = dist_km/5 + gain/600
    h, m = int(T), int((T % 1) * 60)
    print(f'  {name}: {dist_km}/5 + {gain}/600 = {T:.2f}h -> {h}h {m}min')

In [ ]:
# ============================================================
# Visualization: Naismith time breakdown
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Naismith's Rule — Effect of Elevation on Hiking Time",
             fontsize=13, fontweight='bold')

# Plot 1: Time vs Elevation gain (fixed 10km)
elev_range = np.linspace(0, 3000, 300)
axes[0].plot(elev_range, 10/5 + elev_range/600, color='#E74C3C', linewidth=2.5)
axes[0].axvline(x=1053, color='#3498DB', linestyle='--', label='Vihren (+1053m)')
axes[0].axvline(x=2947, color='#27AE60', linestyle='--', label='Rila (+2947m)')
axes[0].set_xlabel('Elevation Gain (m)')
axes[0].set_ylabel('Estimated Time (hours)')
axes[0].set_title('Fixed distance = 10 km')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Time breakdown per track
labels     = ['Rila\n(37.47km,+2947m)', 'Vihren\n(8.87km,+1053m)']
dist_comp  = [37.47/5, 8.87/5]
elev_comp  = [2947/600, 1053/600]
x = np.arange(2)
axes[1].bar(x, dist_comp, label='Distance (d/5)', color='#3498DB', alpha=0.85)
axes[1].bar(x, elev_comp, bottom=dist_comp, label='Elevation (h/600)', color='#E74C3C', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel('Time (hours)')
axes[1].set_title('Naismith Time Breakdown')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
for i, (d, e) in enumerate(zip(dist_comp, elev_comp)):
    axes[1].text(i, d+e+0.1, f'{d+e:.1f}h', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print('Visualization complete!')

---

### 2.4 Summary

| Concept | Role |
|---|---|
| Graph Theory | Models terrain as weighted nodes and edges |
| Haversine | Accurate GPS distance on a sphere |
| Naismith's Rule | Distance + elevation → hiking time |
| Tobler's Function | More accurate alternative (penalizes descents) |
| Dijkstra's Algorithm | Finds minimum-weight path through the graph |

**Next: Section 3 — Data Loading & Graph Construction**

---